# Project 2: Student Academic Performance Analytics
**Industry:** Education / EdTech  
**Curriculum:** Five-Day Engineering Student Training | First Year – First Semester  
**Technologies:** Python 3, NumPy, Pandas  
**Evaluation Standard:** First-Year Engineering Student (Beginner-Friendly, Fully Commented)

---

### **Project Overview & Objectives**
In modern educational institutions, academic performance analytics empowers educators and administrators to proactively identify learning bottlenecks and support struggling students. 

This notebook analyzes student attendance, weekly study hours, home internet connectivity, and assessment scores across three foundational courses:
- **Python Programming (`Python_Marks`)**
- **Applied Mathematics (`Math_Marks`)**
- **Database Management Systems (`DBMS_Marks`)**

#### **Key Analysis Stages:**
1. Data ingestion, structural inspection, and summary statistics.
2. Anomaly detection: finding duplicates, missing values, and invalid score ranges ($<0$ or $>100$).
3. Robust data cleaning and justified imputation strategies.
4. Feature engineering: Total Marks, Average Marks, Grade Bands, and Attendance Categories.
5. Comparative departmental and demographic analytics.
6. Rigorous statistical exploration with **NumPy** (Quartiles, Standard Deviation, IQR Outlier Detection).
7. Five evidence-based, data-backed recommendations for academic support.


### **Setup & Imports**
We import the foundational scientific libraries:
- `numpy`: For numerical computations, arrays, percentiles, and outlier thresholds.
- `pandas`: For loading, cleaning, transforming, and grouping tabular dataset records.


In [1]:
import numpy as np
import pandas as pd

# Set display options for clean and neat tables in the notebook
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Libraries imported successfully!")
print("NumPy Version:", np.__version__)
print("Pandas Version:", pd.__version__)


Libraries imported successfully!
NumPy Version: 2.3.3
Pandas Version: 2.3.2


---
## **Task 1: Load and Inspect the Dataset**
We load the raw CSV file `data/02_student_performance.csv` using Pandas `read_csv()` and inspect its dimensions, columns, and data types.


In [2]:
# Load dataset
# Using relative path compatible with both project root and src/ directory
import os
path = 'data/02_student_performance.csv' if os.path.exists('data/02_student_performance.csv') else '../data/02_student_performance.csv'
df = pd.read_csv(path)

print(f"Dataset Loaded Successfully!")
print(f"Total Rows: {df.shape[0]}, Total Columns: {df.shape[1]}")
print("\n--- Column Data Types ---")
print(df.dtypes)
print("\n--- First 5 Records ---")
df.head()


Dataset Loaded Successfully!
Total Rows: 122, Total Columns: 9

--- Column Data Types ---
Student_ID          object
Name                object
Department          object
Attendance         float64
Study_Hours        float64
Internet_Access     object
Python_Marks       float64
Math_Marks         float64
DBMS_Marks         float64
dtype: object

--- First 5 Records ---
Out[1]: 
  Student_ID              Name        Department  Attendance  Study_Hours Internet_Access  Python_Marks  Math_Marks  DBMS_Marks
0    STU1001  Reyansh Kulkarni       Electronics        78.6          6.7             Yes          50.4        68.8        56.9
1    STU1002         Tanvi Das       Electronics        80.9         10.9             Yes          56.2        68.5        58.8
2    STU1003        Diya Singh  Computer Science        87.9         18.4             Yes          64.7        53.9        60.6
3    STU1004       Tanvi Gupta      Data Science        98.0         15.3             Yes          58.3    

  Student_ID              Name        Department  Attendance  Study_Hours Internet_Access  Python_Marks  Math_Marks  DBMS_Marks
0    STU1001  Reyansh Kulkarni       Electronics        78.6          6.7             Yes          50.4        68.8        56.9
1    STU1002         Tanvi Das       Electronics        80.9         10.9             Yes          56.2        68.5        58.8
2    STU1003        Diya Singh  Computer Science        87.9         18.4             Yes          64.7        53.9        60.6
3    STU1004       Tanvi Gupta      Data Science        98.0         15.3             Yes          58.3        27.1        56.9
4    STU1005    Rahul Deshmukh       Electronics        81.0         14.8              No          55.5        46.9        50.9


In [3]:
# Inspect numerical distributions
print("--- Numerical Feature Statistics (df.describe()) ---")
df.describe()


--- Numerical Feature Statistics (df.describe()) ---
Out[1]: 
       Attendance  Study_Hours  Python_Marks  Math_Marks  DBMS_Marks
count  118.000000   122.000000    122.000000  122.000000  122.000000
mean    79.025424    15.762295     54.468033   53.659836   54.236885
std     12.717833     5.911515     10.614237   12.369568   10.181820
min     50.700000     4.000000     31.300000   -5.000000   28.000000
25%     70.550000    11.625000     46.400000   46.450000   48.825000
50%     80.000000    15.700000     55.550000   54.550000   55.650000
75%     87.550000    19.600000     60.350000   62.550000   60.325000
max     98.000000    31.200000    105.000000   83.500000   74.600000


       Attendance  Study_Hours  Python_Marks  Math_Marks  DBMS_Marks
count  118.000000   122.000000    122.000000  122.000000  122.000000
mean    79.025424    15.762295     54.468033   53.659836   54.236885
std     12.717833     5.911515     10.614237   12.369568   10.181820
min     50.700000     4.000000     31.300000   -5.000000   28.000000
25%     70.550000    11.625000     46.400000   46.450000   48.825000
50%     80.000000    15.700000     55.550000   54.550000   55.650000
75%     87.550000    19.600000     60.350000   62.550000   60.325000
max     98.000000    31.200000    105.000000   83.500000   74.600000


---
## **Task 2: Report Missing Values and Duplicate Student_ID Values**
In academic records, primary keys like `Student_ID` must be unique, and missing records must be quantified before analysis.


In [4]:
# 1. Check for missing values across all columns
missing_summary = df.isnull().sum()
print("Missing Values Per Column:")
print(missing_summary)

# 2. Check for duplicate Student_IDs
duplicate_count = df.duplicated(subset=['Student_ID']).sum()
print(f"\nDuplicate Student_ID count: {duplicate_count}")

# Display duplicate records if any exist
duplicate_rows = df[df.duplicated(subset=['Student_ID'], keep=False)]
if len(duplicate_rows) > 0:
    print("\nSample Duplicate Entries:")
    display(duplicate_rows.sort_values(by='Student_ID'))

# Drop duplicate rows, keeping the first valid record
df_clean = df.drop_duplicates(subset=['Student_ID'], keep='first').copy()
print(f"\nDataset dimensions after removing duplicate Student_IDs: {df_clean.shape}")


Missing Values Per Column:
Student_ID         0
Name               0
Department         0
Attendance         4
Study_Hours        0
Internet_Access    0
Python_Marks       0
Math_Marks         0
DBMS_Marks         0
dtype: int64

Duplicate Student_ID count: 2

Sample Duplicate Entries:
    Student_ID          Name              Department  Attendance  Study_Hours Internet_Access  Python_Marks  Math_Marks  DBMS_Marks
5      STU1006   Aryan Joshi              Mechanical        72.6         28.1              No          57.4        60.8        55.2
120    STU1006   Aryan Joshi              Mechanical        72.6         28.1              No          57.4        60.8        55.2
42     STU1043  Rohan Pandey  Information Technology        73.5         21.0             Yes          57.0        51.6        51.6
121    STU1043  Rohan Pandey  Information Technology        73.5         21.0             Yes          57.0        51.6        51.6

Dataset dimensions after removing duplicate Student_

---
## **Task 3: Handle Missing Attendance Values and Justify Method**
### **Method Chosen: Department-Wise Median Imputation**
**Justification for Beginners:**
1. **Why Median over Mean?** Attendance distributions can have extreme values (e.g. students with very low attendance due to extended leave). The **mean** is easily pulled down by extreme values, whereas the **median** represents the true central tendency of regular students.
2. **Why Group by Department?** Different engineering departments (such as Mechanical vs. Computer Science) have differing lab schedules and attendance patterns. Imputing by department median preserves these natural departmental characteristics.


In [5]:
# Quantify missing attendance before imputation
missing_att_count = df_clean['Attendance'].isnull().sum()
print(f"Missing Attendance values before imputation: {missing_att_count}")

# Compute department-wise median attendance
dept_medians = df_clean.groupby('Department')['Attendance'].transform('median')

# Fill missing attendance with departmental median
df_clean['Attendance'] = df_clean['Attendance'].fillna(dept_medians)

print(f"Missing Attendance values after imputation: {df_clean['Attendance'].isnull().sum()}")
print("\nDepartment-wise Median Attendance values applied:")
print(df_clean.groupby('Department')['Attendance'].median())


Missing Attendance values before imputation: 4
Missing Attendance values after imputation: 0

Department-wise Median Attendance values applied:
Department
Computer Science          80.70
Data Science              82.15
Electronics               82.40
Information Technology    80.70
Mechanical                72.30
Name: Attendance, dtype: float64


---
## **Task 4: Check Score Columns for Values Outside 0–100**
Academic assessment marks must realistically lie between `0` and `100`. Any values outside this range are data-entry errors and must be detected and corrected.


In [6]:
score_columns = ['Python_Marks', 'Math_Marks', 'DBMS_Marks']

# Detect records outside [0, 100]
out_of_bounds = df_clean[((df_clean[score_columns] < 0) | (df_clean[score_columns] > 100)).any(axis=1)]
print(f"Records with out-of-bound scores: {len(out_of_bounds)}")

if len(out_of_bounds) > 0:
    print("\nDetected Out-of-Bounds Records:")
    display(out_of_bounds[['Student_ID', 'Name'] + score_columns])

# Fix: Clip scores to valid boundaries [0.0, 100.0]
for col in score_columns:
    df_clean[col] = df_clean[col].clip(lower=0.0, upper=100.0)

print("\nVerification after cleaning (Min and Max per subject):")
for col in score_columns:
    print(f"  {col:15s} -> Min: {df_clean[col].min():.1f}, Max: {df_clean[col].max():.1f}")


Records with out-of-bound scores: 2

Detected Out-of-Bounds Records:
   Student_ID             Name  Python_Marks  Math_Marks  DBMS_Marks
18    STU1019  Saanvi Kulkarni         105.0        60.4        60.1
50    STU1051       Neha Joshi          70.9        -5.0        33.4

Verification after cleaning (Min and Max per subject):
  Python_Marks    -> Min: 31.3, Max: 100.0
  Math_Marks      -> Min: 0.0, Max: 83.5
  DBMS_Marks      -> Min: 28.0, Max: 74.6


---
## **Task 5: Calculate Total and Average Marks for Each Student**
We create two new columns:
- `Total_Marks`: Sum of `Python_Marks`, `Math_Marks`, and `DBMS_Marks`.
- `Average_Marks`: `Total_Marks / 3` (rounded to 2 decimal places).


In [7]:
# Calculate Total Marks and Average Marks
df_clean['Total_Marks'] = df_clean[score_columns].sum(axis=1)
df_clean['Average_Marks'] = (df_clean['Total_Marks'] / len(score_columns)).round(2)

print("Total and Average Marks calculated successfully!")
df_clean[['Student_ID', 'Name', 'Department', 'Python_Marks', 'Math_Marks', 'DBMS_Marks', 'Total_Marks', 'Average_Marks']].head()


Total and Average Marks calculated successfully!
Out[1]: 
  Student_ID              Name        Department  Python_Marks  Math_Marks  DBMS_Marks  Total_Marks  Average_Marks
0    STU1001  Reyansh Kulkarni       Electronics          50.4        68.8        56.9        176.1          58.70
1    STU1002         Tanvi Das       Electronics          56.2        68.5        58.8        183.5          61.17
2    STU1003        Diya Singh  Computer Science          64.7        53.9        60.6        179.2          59.73
3    STU1004       Tanvi Gupta      Data Science          58.3        27.1        56.9        142.3          47.43
4    STU1005    Rahul Deshmukh       Electronics          55.5        46.9        50.9        153.3          51.10


  Student_ID              Name        Department  Python_Marks  Math_Marks  DBMS_Marks  Total_Marks  Average_Marks
0    STU1001  Reyansh Kulkarni       Electronics          50.4        68.8        56.9        176.1          58.70
1    STU1002         Tanvi Das       Electronics          56.2        68.5        58.8        183.5          61.17
2    STU1003        Diya Singh  Computer Science          64.7        53.9        60.6        179.2          59.73
3    STU1004       Tanvi Gupta      Data Science          58.3        27.1        56.9        142.3          47.43
4    STU1005    Rahul Deshmukh       Electronics          55.5        46.9        50.9        153.3          51.10


---
## **Task 6: Create a Grade Column Using Clearly Defined Score Ranges**
We categorize performance into standard institutional academic grade bands using `pd.cut()`:
- **A (Outstanding):** $[85 - 100]$
- **B (Good):** $[70 - 85)$
- **C (Satisfactory):** $[55 - 70)$
- **D (Pass):** $[40 - 55)$
- **F (Fail):** $[0 - 40)$


In [8]:
# Define grade boundaries and labels
grade_bins = [0, 40, 55, 70, 85, 100]
grade_labels = ['F (Fail)', 'D (Pass)', 'C (Satisfactory)', 'B (Good)', 'A (Outstanding)']

df_clean['Grade'] = pd.cut(
    df_clean['Average_Marks'],
    bins=grade_bins,
    labels=grade_labels,
    include_lowest=True,
    right=False
)

# Summarize grade distribution
grade_dist = df_clean['Grade'].value_counts().sort_index().reset_index()
grade_dist.columns = ['Grade', 'Student_Count']
grade_dist['Percentage (%)'] = (grade_dist['Student_Count'] / len(df_clean) * 100).round(2)

print("Grade Distribution Summary:")
display(grade_dist)


Grade Distribution Summary:
              Grade  Student_Count  Percentage (%)
0          F (Fail)              6            5.00
1          D (Pass)             58           48.33
2  C (Satisfactory)             53           44.17
3          B (Good)              3            2.50
4   A (Outstanding)              0            0.00


---
## **Task 7: Calculate Department-Wise Average Marks**
We use `groupby('Department').agg()` to evaluate how different engineering branches perform across all subjects.


In [9]:
dept_performance = df_clean.groupby('Department').agg(
    Student_Count=('Student_ID', 'count'),
    Avg_Overall_Marks=('Average_Marks', 'mean'),
    Avg_Python=('Python_Marks', 'mean'),
    Avg_Math=('Math_Marks', 'mean'),
    Avg_DBMS=('DBMS_Marks', 'mean'),
    Avg_Attendance=('Attendance', 'mean')
).round(2).sort_values(by='Avg_Overall_Marks', ascending=False)

print("Department-Wise Performance Comparison:")
display(dept_performance)


Department-Wise Performance Comparison:
                        Student_Count  Avg_Overall_Marks  Avg_Python  Avg_Math  Avg_DBMS  Avg_Attendance
Department                                                                                              
Electronics                        22              56.51       58.41     54.75     56.38           82.05
Mechanical                         22              54.70       54.17     54.13     55.81           73.73
Computer Science                   23              54.21       56.93     53.39     52.30           78.62
Data Science                       24              53.49       50.44     53.33     56.68           80.47
Information Technology             29              52.23       52.73     52.96     50.99           79.36


---
## **Task 8: Find the Top 10 Students by Average Marks**
We rank the student cohort by `Average_Marks` in descending order to identify the top 10 academic achievers.


In [10]:
top_10 = df_clean.sort_values(by='Average_Marks', ascending=False).head(10)

print("Top 10 High-Performing Students:")
display(top_10[['Student_ID', 'Name', 'Department', 'Attendance', 'Study_Hours', 'Average_Marks', 'Grade']])


Top 10 High-Performing Students:
    Student_ID             Name              Department  Attendance  Study_Hours  Average_Marks             Grade
18     STU1019  Saanvi Kulkarni        Computer Science        82.5         13.0          73.50          B (Good)
78     STU1079       Ananya Das  Information Technology        95.8         15.3          70.70          B (Good)
17     STU1018   Karan Deshmukh             Electronics        92.5         23.1          70.37          B (Good)
113    STU1114       Rahul Iyer        Computer Science        98.0         31.2          69.83  C (Satisfactory)
38     STU1039      Karan Reddy              Mechanical        98.0         25.5          69.60  C (Satisfactory)
116    STU1117      Neha Mishra        Computer Science        98.0         22.9          69.20  C (Satisfactory)
12     STU1013    Saanvi Chopra              Mechanical        72.3         26.9          67.57  C (Satisfactory)
33     STU1034      Vivaan Iyer             Electronics

---
## **Task 9: Identify Students Below 40% Average Marks (At-Risk)**
Students with $<40\%$ average marks are at risk of failing the semester and require immediate remedial attention.


In [11]:
at_risk = df_clean[df_clean['Average_Marks'] < 40.0].sort_values(by='Average_Marks')

print(f"Total Students Below 40% (At-Risk): {len(at_risk)}")
print(f"Percentage of Student Body: {(len(at_risk) / len(df_clean)) * 100:.2f}%\n")

display(at_risk[['Student_ID', 'Name', 'Department', 'Attendance', 'Study_Hours', 'Internet_Access', 'Average_Marks', 'Grade']])


Total Students Below 40% (At-Risk): 6
Percentage of Student Body: 5.00%

   Student_ID          Name              Department  Attendance  Study_Hours Internet_Access  Average_Marks     Grade
75    STU1076   Tanvi Reddy        Computer Science        70.5          6.4              No          30.47  F (Fail)
19    STU1020   Pooja Patel            Data Science        61.5          7.9             Yes          31.40  F (Fail)
50    STU1051    Neha Joshi        Computer Science        80.1          4.6             Yes          34.77  F (Fail)
71    STU1072   Rahul Mehta        Computer Science        52.7         11.4              No          35.30  F (Fail)
6     STU1007    Diya Reddy  Information Technology        53.6         10.9              No          35.40  F (Fail)
66    STU1067  Aadhya Patel  Information Technology        81.1          8.7              No          39.37  F (Fail)


---
## **Task 10: Compare Average Marks for Different Attendance Bands**
We segment student attendance into institutional policy bands using `pd.cut()`:
- `<60%`: Critical Shortage
- `60-75%`: Shortage / Warning Zone
- `75-85%`: Satisfactory Attendance
- `85-100%`: High Attendance


In [12]:
att_bins = [0, 60, 75, 85, 100]
att_labels = ['<60% (Critical)', '60-75% (Shortage)', '75-85% (Satisfactory)', '85-100% (High)']

df_clean['Attendance_Band'] = pd.cut(
    df_clean['Attendance'],
    bins=att_bins,
    labels=att_labels,
    include_lowest=True,
    right=False
)

att_summary = df_clean.groupby('Attendance_Band', observed=False).agg(
    Student_Count=('Student_ID', 'count'),
    Avg_Marks=('Average_Marks', 'mean'),
    Median_Marks=('Average_Marks', 'median'),
    Min_Marks=('Average_Marks', 'min'),
    Max_Marks=('Average_Marks', 'max')
).round(2)

print("Academic Performance Across Attendance Bands:")
display(att_summary)


Academic Performance Across Attendance Bands:
                       Student_Count  Avg_Marks  Median_Marks  Min_Marks  Max_Marks
Attendance_Band                                                                    
<60% (Critical)                   12      45.59         44.50      35.30      59.43
60-75% (Shortage)                 31      52.83         53.40      30.47      67.57
75-85% (Satisfactory)             40      54.21         54.47      34.77      73.50
85-100% (High)                    37      57.80         57.10      41.50      70.70


---
## **Task 11: Analyze Relationship Between Study Hours and Average Marks**
We compute the Pearson correlation coefficient between weekly self-study hours and average marks, followed by grouping into study tiers.


In [13]:
corr_study = df_clean['Study_Hours'].corr(df_clean['Average_Marks'])
print(f"Pearson Correlation (Study Hours vs. Average Marks): {corr_study:.3f}")

# Group into study hour bands
hour_bins = [0, 10, 20, 35]
hour_labels = ['Low (<10 hrs/wk)', 'Medium (10-20 hrs/wk)', 'High (>20 hrs/wk)']

df_clean['Study_Hour_Band'] = pd.cut(
    df_clean['Study_Hours'],
    bins=hour_bins,
    labels=hour_labels,
    include_lowest=True,
    right=False
)

study_summary = df_clean.groupby('Study_Hour_Band', observed=False).agg(
    Student_Count=('Student_ID', 'count'),
    Avg_Marks=('Average_Marks', 'mean'),
    Median_Marks=('Average_Marks', 'median')
).round(2)

print("\nAcademic Performance by Study Hours Band:")
display(study_summary)

print("\nMethodology Reminder:")
print("Correlation measures statistical association, NOT causation. Other factors such as prior preparation and teaching quality also contribute.")


Pearson Correlation (Study Hours vs. Average Marks): 0.620

Academic Performance by Study Hours Band:
                       Student_Count  Avg_Marks  Median_Marks
Study_Hour_Band                                              
Low (<10 hrs/wk)                  17      44.72         45.33
Medium (10-20 hrs/wk)             77      53.96         54.47
High (>20 hrs/wk)                 26      60.64         60.95

Methodology Reminder:
Correlation measures statistical association, NOT causation. Other factors such as prior preparation and teaching quality also contribute.


---
## **Task 12: Compare Results for Students With and Without Internet Access**
We assess the impact of home digital access on overall and subject-specific marks.


In [14]:
internet_analysis = df_clean.groupby('Internet_Access').agg(
    Student_Count=('Student_ID', 'count'),
    Avg_Overall_Marks=('Average_Marks', 'mean'),
    Median_Overall_Marks=('Average_Marks', 'median'),
    Avg_Python=('Python_Marks', 'mean'),
    Avg_Math=('Math_Marks', 'mean'),
    Avg_DBMS=('DBMS_Marks', 'mean')
).round(2)

print("Performance Comparison by Internet Access:")
display(internet_analysis)

gap = internet_analysis.loc['Yes', 'Avg_Overall_Marks'] - internet_analysis.loc['No', 'Avg_Overall_Marks']
print(f"\nPerformance Disparity: Students with internet access score {gap:.2f} marks higher on average.")


Performance Comparison by Internet Access:
                 Student_Count  Avg_Overall_Marks  Median_Overall_Marks  Avg_Python  Avg_Math  Avg_DBMS
Internet_Access                                                                                        
No                          24              49.32                 50.36       49.82      48.3     49.85
Yes                         96              55.29                 55.50       55.52      55.0     55.35

Performance Disparity: Students with internet access score 5.97 marks higher on average.


---
## **Task 13: Calculate Median, Standard Deviation and Quartiles of Python_Marks using NumPy**
Using purely NumPy array methods (`np.median`, `np.std`, `np.percentile`), we inspect the statistical dispersion of Python examination scores.


In [15]:
# Extract NumPy 1D array
python_scores = df_clean['Python_Marks'].to_numpy()

# Calculate descriptive statistics with NumPy
np_mean = np.mean(python_scores)
np_median = np.median(python_scores)
np_std = np.std(python_scores)
np_var = np.var(python_scores)
q1 = np.percentile(python_scores, 25)
q2 = np.percentile(python_scores, 50)
q3 = np.percentile(python_scores, 75)

print("=== NumPy Statistical Metrics for Python_Marks ===")
print(f"Mean Score              : {np_mean:.2f}")
print(f"Median Score (Q2)       : {np_median:.2f}")
print(f"Standard Deviation      : {np_std:.2f}")
print(f"Variance                : {np_var:.2f}")
print(f"First Quartile (Q1, 25%): {q1:.2f}")
print(f"Second Quartile (Q2, 50%): {q2:.2f}")
print(f"Third Quartile (Q3, 75%): {q3:.2f}")


=== NumPy Statistical Metrics for Python_Marks ===
Mean Score              : 54.38
Median Score (Q2)       : 55.25
Standard Deviation      : 10.46
Variance                : 109.47
First Quartile (Q1, 25%): 46.35
Second Quartile (Q2, 50%): 55.25
Third Quartile (Q3, 75%): 60.48


---
## **Task 14: Identify Potential Score Outliers using IQR (Interquartile Range)**
We apply the standard statistical Interquartile Range (IQR) rule:
- $\text{IQR} = Q_3 - Q_1$
- $\text{Lower Fence} = Q_1 - 1.5 \times \text{IQR}$
- $\text{Upper Fence} = Q_3 + 1.5 \times \text{IQR}$
Any score below the lower fence or above the upper fence is flagged as a statistical outlier.


In [16]:
# Calculate IQR and fences
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

print(f"Interquartile Range (IQR) : {iqr:.2f}")
print(f"Lower Fence (Q1 - 1.5*IQR): {lower_fence:.2f}")
print(f"Upper Fence (Q3 + 1.5*IQR): {upper_fence:.2f}")

# Filter outliers
outliers = df_clean[(df_clean['Python_Marks'] < lower_fence) | (df_clean['Python_Marks'] > upper_fence)]
print(f"\nNumber of detected outliers in Python_Marks: {len(outliers)}")

if len(outliers) > 0:
    print("\nIdentified Outliers:")
    display(outliers[['Student_ID', 'Name', 'Department', 'Python_Marks', 'Average_Marks']])
else:
    print("No outliers detected beyond fences.")


Interquartile Range (IQR) : 14.12
Lower Fence (Q1 - 1.5*IQR): 25.16
Upper Fence (Q3 + 1.5*IQR): 81.66

Number of detected outliers in Python_Marks: 1

Identified Outliers:
   Student_ID             Name        Department  Python_Marks  Average_Marks
18    STU1019  Saanvi Kulkarni  Computer Science         100.0           73.5


---
## **Task 15: Five Evidence-Based Recommendations for Academic Support**
Based strictly on empirical evidence derived from our dataset:

1. **Early Attendance Warning System:**  
   Students in the critical attendance band ($<60\%$) averaged **45.59 marks**, in sharp contrast to **57.80 marks** for students with $>85\%$ attendance. The institution should introduce automated SMS/portal alerts at the $75\%$ threshold to prevent compounding deficits.

2. **Targeted Remedial Support for At-Risk Students:**  
   A total of **6 students** scored below the passing mark ($<40\%$ average). Faculty advisors should pair each at-risk student with dedicated teaching assistants for weekend concept reinforcement in Python and Mathematics.

3. **Campus Digital Equity Initiative:**  
   Students lacking home internet access scored **~6 marks lower** ($49.32$ vs $55.29$). The college should provide 24/7 library computer lab access, loaner laptops, and pre-loaded offline video lectures.

4. **Structured Peer Study Circles:**  
   Weekly self-study hours demonstrated a strong positive association with marks ($r = 0.62$). Students studying $<10$ hours/week averaged only $44.72$ marks. Establishing guided peer-study halls can foster productive study habits.

5. **Inter-Departmental Teaching Collaboration:**  
   Departmental averages ranged from $52.23$ (IT) to $56.51$ (Electronics). Departments should organize monthly pedagogical forums where high-performing faculty share active learning techniques and coding lab exercises.


In [17]:
# Save the cleaned dataset for LMS evaluation
out_path = 'data/02_student_performance_cleaned.csv' if os.path.exists('data') else '../data/02_student_performance_cleaned.csv'
df_clean.to_csv(out_path, index=False)
print(f"Cleaned dataset saved successfully to: {out_path}")


Cleaned dataset saved successfully to: data/02_student_performance_cleaned.csv
